# Türkçe Afet Tweet Sınıflandırması — QLoRA Fine-Tuning
**Mustafa Kırpık** | BDM Dönem Projesi Finali | Selçuk Üniversitesi

Bu notebook 5 sınıflı Türkçe afet tweet sınıflandırması için **QLoRA (4-bit NF4)** ile fine-tuning yapar.
Modeller: SmolLM2-360M, TinyLlama-1.1B, Qwen2.5-1.5B, Gemma-2-2B-it.

**Saliha Göçergi LoRA**, ben **QLoRA** kullanacağım → karşılaştırmalı benchmark.

### Çalışma Akışı
1. **Setup** — kütüphaneler, GPU kontrolü, seed ayarı
2. **Veri Yükleme** — Drive'a yüklediğin train/val/test CSV'leri
3. **Zero-Shot Baseline** — her 4 model için fine-tuning ÖNCESİ skor
4. **QLoRA Fine-Tuning** — sırayla 4 model için
5. **Değerlendirme** — Accuracy, Macro-F1, Weighted-F1, Confusion Matrix
6. **Çoklu Seed** — 3 seed (42, 123, 2023) ile tekrarlanabilirlik
7. **Hata Analizi** — her modelden en az 5 başarısız örnek
8. **Model Kartları** — her fine-tuned model için

## 1. Setup

In [ ]:
# Colab GPU kontrolü
!nvidia-smi

In [ ]:
# Kütüphaneler
!pip install -q -U transformers==4.44.2 peft==0.12.0 bitsandbytes==0.43.3 accelerate==0.33.0 datasets==2.21.0 trl==0.10.1 scikit-learn pandas matplotlib seaborn

In [ ]:
import os, json, random, time, gc
import numpy as np
import pandas as pd
import torch
from pathlib import Path

from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          BitsAndBytesConfig, TrainingArguments, Trainer,
                          DataCollatorWithPadding, set_seed)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from datasets import Dataset
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix)
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
set_seed(SEED)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-"}')

## 2. Veri Yükleme

**ADIM:** `train.csv`, `val.csv`, `test.csv` dosyalarını Google Drive'ında bir klasöre yükle (örn. `MyDrive/bdm_proje/data/`).
Sonra aşağıdaki hücrede yolu güncelle.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA_DIR = '/content/drive/MyDrive/bdm_proje/data'   # ← BURAYI KENDİ KLASÖRÜNLE DEĞİŞTİR
OUT_DIR  = '/content/drive/MyDrive/bdm_proje/outputs'
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
val_df   = pd.read_csv(f'{DATA_DIR}/val.csv')
test_df  = pd.read_csv(f'{DATA_DIR}/test.csv')

print('TRAIN:', len(train_df), '| VAL:', len(val_df), '| TEST:', len(test_df))
print('\nSınıf dağılımı (train):')
print(train_df['label_name'].value_counts())

In [ ]:
LABEL_NAMES = ['yardim_talebi', 'kayip_bildirimi', 'altyapi_hasari',
               'bagis_koordinasyon', 'diger_ilgisiz']
ID2LABEL = {i: n for i, n in enumerate(LABEL_NAMES)}
LABEL2ID = {n: i for i, n in enumerate(LABEL_NAMES)}
NUM_LABELS = len(LABEL_NAMES)

def to_dataset(df):
    return Dataset.from_pandas(df[['text', 'label']].reset_index(drop=True))

train_ds = to_dataset(train_df)
val_ds   = to_dataset(val_df)
test_ds  = to_dataset(test_df)
print(train_ds)

## 3. Yardımcı Fonksiyonlar

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy':    accuracy_score(labels, preds),
        'macro_f1':    f1_score(labels, preds, average='macro', zero_division=0),
        'weighted_f1': f1_score(labels, preds, average='weighted', zero_division=0),
    }

def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def gpu_mem_gb():
    if not torch.cuda.is_available():
        return 0
    return torch.cuda.max_memory_allocated() / 1024**3

## 4. Model Konfigürasyonları

Hocanın istediği 4 modelden 3'ü zorunlu. Ben hepsini hazırladım, sen Colab GPU limitine göre 3 veya 4'ü çalıştırabilirsin.
**Sıralama:** önce küçükten büyüğe (RAM ısınma sırası).

In [ ]:
MODEL_CONFIGS = {
    'SmolLM2-360M':   {'name': 'HuggingFaceTB/SmolLM2-360M-Instruct',  'lr': 3e-4, 'r': 16, 'alpha': 32, 'target': ['q_proj','k_proj','v_proj','o_proj']},
    'TinyLlama-1.1B': {'name': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',    'lr': 2e-4, 'r': 16, 'alpha': 32, 'target': ['q_proj','k_proj','v_proj','o_proj']},
    'Qwen2.5-1.5B':   {'name': 'Qwen/Qwen2.5-1.5B-Instruct',            'lr': 2e-4, 'r': 16, 'alpha': 32, 'target': ['q_proj','k_proj','v_proj','o_proj']},
    'Gemma-2-2B':     {'name': 'google/gemma-2-2b-it',                   'lr': 1e-4, 'r': 16, 'alpha': 32, 'target': ['q_proj','k_proj','v_proj','o_proj']},
}

# QLoRA quantization config (4-bit NF4 + Double Quantization)
BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Eğitim hiperparametreleri (ortak)
EPOCHS = 3
BATCH_SIZE = 8
GRAD_ACCUM = 2
MAX_LEN = 256

## 5. Zero-Shot Baseline

Her modelin fine-tuning ÖNCESİ performansı (rapor zorunluluğu). Sınıflandırma görevi için prompt-based zero-shot:
modele bir prompt veriyoruz, ilk üretilen kelimeyi etiket olarak alıyoruz.

In [ ]:
from transformers import AutoModelForCausalLM

ZS_PROMPT_TEMPLATE = '''Aşağıdaki Türkçe tweet'i şu 5 kategoriden BİRİNE ata.\nKategoriler:\n0 = yardim_talebi (acil yardım, kurtarma, malzeme talebi)\n1 = kayip_bildirimi (kişiye ulaşılamıyor, kayıp arama)\n2 = altyapi_hasari (yıkık bina, kapalı yol, elektrik/su kesintisi)\n3 = bagis_koordinasyon (bağış, IBAN, gönüllü, lojistik)\n4 = diger_ilgisiz (haber, taziye, eleştiri, off-topic)\n\nTweet: {text}\n\nCevap (sadece bir rakam, 0/1/2/3/4):'''

def zero_shot_eval(model_name, texts, labels, max_samples=200):
    print(f'\n[ZERO-SHOT] {model_name} | {min(len(texts), max_samples)} örnek')
    tok = AutoTokenizer.from_pretrained(model_name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    mdl = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=BNB_CONFIG, device_map='auto')
    mdl.eval()

    preds = []
    t0 = time.time()
    for i in range(min(len(texts), max_samples)):
        prompt = ZS_PROMPT_TEMPLATE.format(text=texts[i][:400])
        inp = tok(prompt, return_tensors='pt', truncation=True, max_length=512).to(device)
        with torch.no_grad():
            out = mdl.generate(**inp, max_new_tokens=5, do_sample=False, temperature=1.0,
                              pad_token_id=tok.pad_token_id)
        ans = tok.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        # İlk rakamı yakala
        digit = next((c for c in ans if c.isdigit() and int(c) in [0,1,2,3,4]), '4')
        preds.append(int(digit))
    elapsed = time.time() - t0

    n = len(preds)
    truth = labels[:n]
    acc = accuracy_score(truth, preds)
    mf1 = f1_score(truth, preds, average='macro', zero_division=0)
    wf1 = f1_score(truth, preds, average='weighted', zero_division=0)
    print(f'  Acc={acc:.4f} | MacroF1={mf1:.4f} | WeightedF1={wf1:.4f} | {elapsed:.1f}s ({elapsed/n*1000:.0f}ms/örnek)')

    del mdl, tok; free_memory()
    return {'accuracy': acc, 'macro_f1': mf1, 'weighted_f1': wf1, 'inference_ms': elapsed/n*1000}

# Zero-shot sonuçlarını topla
ZS_RESULTS = {}
test_texts = test_df['text'].tolist()
test_labels = test_df['label'].tolist()

for mkey, cfg in MODEL_CONFIGS.items():
    try:
        ZS_RESULTS[mkey] = zero_shot_eval(cfg['name'], test_texts, test_labels, max_samples=200)
    except Exception as e:
        print(f'❌ {mkey} hata: {e}')
        ZS_RESULTS[mkey] = {'error': str(e)}

pd.DataFrame(ZS_RESULTS).T.to_csv(f'{OUT_DIR}/zero_shot_results.csv')
print('\n=== ZERO-SHOT SONUÇLARI ===')
print(pd.DataFrame(ZS_RESULTS).T)

## 6. QLoRA Fine-Tuning — Tek Model İçin Hücre

Aşağıdaki hücreyi 4 model için sırayla çalıştır. Her model bittikten sonra `free_memory()` ile RAM temizleniyor.
Adımlar tekrarlanabilir olsun diye **3 seed (42, 123, 2023)** ile çalışıyoruz.

In [ ]:
def tokenize_dataset(ds, tok, max_len=MAX_LEN):
    def _fn(batch):
        return tok(batch['text'], truncation=True, max_length=max_len, padding=False)
    return ds.map(_fn, batched=True, remove_columns=['text'])

def fine_tune_qlora(model_key, seed=42, run_eval_only_at_end=True):
    cfg = MODEL_CONFIGS[model_key]
    model_name = cfg['name']
    print(f'\n{"="*70}\n🚀 {model_key} | seed={seed}\n{"="*70}')

    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(model_name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = 'right'

    tr_tok = tokenize_dataset(train_ds, tok)
    va_tok = tokenize_dataset(val_ds, tok)
    te_tok = tokenize_dataset(test_ds, tok)

    # Sınıflandırma için sequence classification head
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=NUM_LABELS,
        quantization_config=BNB_CONFIG, device_map='auto',
        id2label=ID2LABEL, label2id=LABEL2ID,
    )
    model.config.pad_token_id = tok.pad_token_id
    model = prepare_model_for_kbit_training(model)

    lora_cfg = LoraConfig(
        task_type=TaskType.SEQ_CLS, r=cfg['r'], lora_alpha=cfg['alpha'],
        lora_dropout=0.05, bias='none', target_modules=cfg['target'],
        modules_to_save=['score'],  # classification head'i de eğit
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()

    out_dir = f'{OUT_DIR}/{model_key}_seed{seed}'
    args = TrainingArguments(
        output_dir=out_dir,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE*2,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=cfg['lr'],
        warmup_ratio=0.05,
        lr_scheduler_type='cosine',
        weight_decay=0.01,
        optim='paged_adamw_8bit',  # QLoRA'nın Paged Optimizer'ı
        logging_steps=50,
        eval_strategy='epoch',
        save_strategy='no',
        seed=seed,
        bf16=True,
        report_to='none',
        gradient_checkpointing=True,
    )
    collator = DataCollatorWithPadding(tokenizer=tok)
    trainer = Trainer(
        model=model, args=args,
        train_dataset=tr_tok, eval_dataset=va_tok,
        tokenizer=tok, data_collator=collator,
        compute_metrics=compute_metrics,
    )

    torch.cuda.reset_peak_memory_stats() if torch.cuda.is_available() else None
    t0 = time.time()
    trainer.train()
    train_time = time.time() - t0

    # Test
    t1 = time.time()
    test_metrics = trainer.evaluate(te_tok)
    test_pred = trainer.predict(te_tok)
    test_time = time.time() - t1
    n_test = len(te_tok)
    inference_ms = (test_time / n_test) * 1000

    preds = np.argmax(test_pred.predictions, axis=-1)
    truth = test_pred.label_ids

    result = {
        'model': model_key, 'seed': seed,
        'accuracy':    accuracy_score(truth, preds),
        'macro_f1':    f1_score(truth, preds, average='macro', zero_division=0),
        'weighted_f1': f1_score(truth, preds, average='weighted', zero_division=0),
        'train_time_s':  round(train_time, 1),
        'inference_ms':  round(inference_ms, 2),
        'gpu_mem_gb':    round(gpu_mem_gb(), 2),
        'trainable_params': sum(p.numel() for p in model.parameters() if p.requires_grad),
        'classification_report': classification_report(truth, preds, target_names=LABEL_NAMES, zero_division=0, output_dict=True),
        'confusion_matrix': confusion_matrix(truth, preds).tolist(),
    }

    # Sonuçları kaydet
    with open(f'{OUT_DIR}/{model_key}_seed{seed}_result.json', 'w') as f:
        json.dump(result, f, indent=2, default=str)

    # LoRA adaptörleri kaydet (model_kartı için)
    model.save_pretrained(f'{OUT_DIR}/{model_key}_seed{seed}_adapter')
    tok.save_pretrained(f'{OUT_DIR}/{model_key}_seed{seed}_adapter')

    print(f'\n✓ {model_key} | seed={seed}: Acc={result["accuracy"]:.4f} | MF1={result["macro_f1"]:.4f} | WF1={result["weighted_f1"]:.4f}')
    print(f'  Eğitim süresi: {train_time:.1f}s | Inference: {inference_ms:.2f}ms/örnek | GPU mem: {gpu_mem_gb():.2f}GB')

    del model, trainer, tok; free_memory()
    return result

### Tek seed ile dene — Önce bir model bitsin sonra diğerleri
Önerim: önce **SmolLM2-360M** ile pipeline'ı test et, sorunsuz çalışırsa diğer modellere geç.

In [ ]:
# İlk pipeline testi: SmolLM2 / seed=42
ALL_RESULTS = []
r = fine_tune_qlora('SmolLM2-360M', seed=42)
ALL_RESULTS.append(r)

In [ ]:
# Geri kalan 3 model (zorunlu set), seed=42
for mkey in ['TinyLlama-1.1B', 'Qwen2.5-1.5B', 'Gemma-2-2B']:
    try:
        r = fine_tune_qlora(mkey, seed=42)
        ALL_RESULTS.append(r)
    except Exception as e:
        print(f'❌ {mkey} HATA: {e}')
        ALL_RESULTS.append({'model': mkey, 'seed': 42, 'error': str(e)})

In [ ]:
# 3 seed × 4 model = 12 koşu — TÜM KOMBİNASYONLAR (uzun, opsiyonel)
# Hocanın "3 farklı seed ile tekrarlanıp ortalama ± std raporlanacak" şartı için.
# Eğer Colab kotası kısıtlıysa sadece SmolLM2 ve Qwen için 3 seed yapabilirsin.

FULL_SEEDS = [42, 123, 2023]
for seed in FULL_SEEDS[1:]:    # 42 zaten yapıldı yukarıda
    for mkey in MODEL_CONFIGS.keys():
        try:
            r = fine_tune_qlora(mkey, seed=seed)
            ALL_RESULTS.append(r)
        except Exception as e:
            print(f'❌ {mkey} seed={seed} HATA: {e}')
            ALL_RESULTS.append({'model': mkey, 'seed': seed, 'error': str(e)})

# Sonuçları toplu kaydet
with open(f'{OUT_DIR}/all_results.json', 'w') as f:
    json.dump(ALL_RESULTS, f, indent=2, default=str)

## 7. Standart Sonuç Tablosu (Hocanın Formatı)

In [ ]:
summary_rows = []
for r in ALL_RESULTS:
    if 'error' in r: continue
    summary_rows.append({
        'Model': r['model'], 'Seed': r['seed'],
        'Accuracy': round(r['accuracy'], 4),
        'Macro-F1': round(r['macro_f1'], 4),
        'Weighted-F1': round(r['weighted_f1'], 4),
        'Train (s)':       r['train_time_s'],
        'Inference (ms)':  r['inference_ms'],
        'GPU mem (GB)':    r['gpu_mem_gb'],
        'Trainable params': r['trainable_params'],
    })
summary = pd.DataFrame(summary_rows)
summary.to_csv(f'{OUT_DIR}/summary_per_run.csv', index=False)
print(summary.to_string(index=False))

# Çoklu seed ortalama ± std
agg = summary.groupby('Model').agg({
    'Accuracy': ['mean', 'std'],
    'Macro-F1': ['mean', 'std'],
    'Weighted-F1': ['mean', 'std'],
    'Train (s)': 'mean',
    'Inference (ms)': 'mean',
    'GPU mem (GB)': 'mean',
}).round(4)
agg.to_csv(f'{OUT_DIR}/summary_mean_std.csv')
print('\n=== ORTALAMA ± STD ===')
print(agg)

## 8. Confusion Matrix Görselleştirme

In [ ]:
fig, axes = plt.subplots(1, len(MODEL_CONFIGS), figsize=(20, 4))
if len(MODEL_CONFIGS) == 1:
    axes = [axes]

for ax, (mkey, _) in zip(axes, MODEL_CONFIGS.items()):
    runs = [r for r in ALL_RESULTS if r.get('model')==mkey and r.get('seed')==42]
    if not runs: continue
    cm = np.array(runs[0]['confusion_matrix'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, cbar=False)
    ax.set_title(mkey)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Hata Analizi (her modelden en az 5 başarısız örnek)

In [ ]:
# Her model için yeniden tahmin yapıp hatalı örnekleri çıkar
from peft import PeftModel

def load_and_predict_test(model_key, seed=42):
    cfg = MODEL_CONFIGS[model_key]
    tok = AutoTokenizer.from_pretrained(cfg['name'])
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    base = AutoModelForSequenceClassification.from_pretrained(
        cfg['name'], num_labels=NUM_LABELS, quantization_config=BNB_CONFIG, device_map='auto',
        id2label=ID2LABEL, label2id=LABEL2ID,
    )
    base.config.pad_token_id = tok.pad_token_id
    mdl = PeftModel.from_pretrained(base, f'{OUT_DIR}/{model_key}_seed{seed}_adapter')
    mdl.eval()

    preds = []
    for text in test_df['text']:
        inp = tok(text, return_tensors='pt', truncation=True, max_length=MAX_LEN).to(device)
        with torch.no_grad():
            out = mdl(**inp)
        preds.append(int(out.logits.argmax(-1)))
    del mdl, base, tok; free_memory()
    return preds

errors_by_model = {}
for mkey in MODEL_CONFIGS.keys():
    try:
        preds = load_and_predict_test(mkey)
        df_err = test_df.copy()
        df_err['pred'] = preds
        df_err['pred_name'] = df_err['pred'].map(ID2LABEL)
        df_err = df_err[df_err['label'] != df_err['pred']].head(10)
        errors_by_model[mkey] = df_err
        df_err.to_csv(f'{OUT_DIR}/errors_{mkey}.csv', index=False)
        print(f'\n=== {mkey} — 10 başarısız örnek ===')
        for _, row in df_err.iterrows():
            print(f'  TRUE={row["label_name"]:20s} PRED={row["pred_name"]:20s} | {row["text"][:100]}')
    except Exception as e:
        print(f'{mkey} hata: {e}')

## 10. Model Kartları

In [ ]:
def write_model_card(model_key, result):
    md = f'''# Model Kartı — {model_key} (QLoRA Fine-tuned)

## Model Bilgisi
- **Temel model:** {MODEL_CONFIGS[model_key]["name"]}
- **Görev:** 5-sınıflı Türkçe afet tweet sınıflandırması
- **Fine-tuning yöntemi:** QLoRA (4-bit NF4 + Double Quantization + Paged Optimizer)
- **Eğitim seed:** {result["seed"]}

## Eğitim Verisi
- **Kaynak:** Toraman et al. (2023) Tweets Under the Rubble + Kaggle Türkiye Deprem Yardımlaşma + zayıf etiketlenmiş depremGun5 + sentetik veri
- **Boyut:** ~{len(train_df):,} train / {len(val_df):,} val / {len(test_df):,} test
- **Dil:** Türkçe
- **Lisans:** Bilimsel kullanım (kişisel veriler anonimleştirilmiştir)

## Hiperparametreler
- LoRA r={MODEL_CONFIGS[model_key]["r"]}, alpha={MODEL_CONFIGS[model_key]["alpha"]}
- Learning rate: {MODEL_CONFIGS[model_key]["lr"]}, Epoch: {EPOCHS}, Batch: {BATCH_SIZE}
- Optimizer: Paged AdamW 8-bit, Scheduler: Cosine, Warmup: 5%
- Max sequence length: {MAX_LEN}

## Performans
- **Accuracy:** {result["accuracy"]:.4f}
- **Macro-F1:** {result["macro_f1"]:.4f}
- **Weighted-F1:** {result["weighted_f1"]:.4f}
- **Inference:** {result["inference_ms"]:.2f} ms/örnek
- **GPU bellek:** {result["gpu_mem_gb"]:.2f} GB (eğitim peak)
- **Eğitilebilir parametre:** {result["trainable_params"]:,}

## Bilinen Limitasyonlar ve Bias Riskleri
- Eğitim verisinin bir bölümü (sentetik) template-based üretildi; bu sınıflarda model template pattern'lerine fazla uyabilir.
- Pseudo-etiketleme (~%70-80 doğruluk) ile genişletilmiş veri kullanıldı; bazı etiket gürültüsü mevcut.
- Twitter'ın demografik bias'ı (yaş, kentsel/kırsal, dil) test edilmedi.
- Off-topic yardım çağrıları (örn. ekonomik kriz) tweet'leri ile karışıklık olabilir.

## Çalıştırma Gereksinimleri
- GPU VRAM: en az 8 GB (4-bit), 16 GB önerilir
- Python 3.10+, transformers 4.44+, peft 0.12+, bitsandbytes 0.43+
- Kullanım: AutoTokenizer + AutoModelForSequenceClassification + PEFT adapter
'''
    with open(f'{OUT_DIR}/MODELCARD_{model_key}.md', 'w', encoding='utf-8') as f:
        f.write(md)
    return md

for r in ALL_RESULTS:
    if 'error' in r: continue
    if r['seed'] != 42: continue   # her model için sadece bir kartı seed=42'den yaz
    write_model_card(r['model'], r)
print('Model kartları yazıldı.')

## 11. GitHub'a Yükleme — README

Tüm çıktıları indirip GitHub repo'na yükleyebilirsin:
- `outputs/all_results.json`
- `outputs/summary_per_run.csv`, `summary_mean_std.csv`
- `outputs/confusion_matrices.png`
- `outputs/errors_*.csv`
- `outputs/MODELCARD_*.md`
- Adaptör klasörleri (`*_adapter/`) — opsiyonel, HuggingFace Hub'a da yükleyebilirsin

### requirements.txt için içerik:
```
transformers==4.44.2
peft==0.12.0
bitsandbytes==0.43.3
accelerate==0.33.0
datasets==2.21.0
trl==0.10.1
scikit-learn
pandas
matplotlib
seaborn
```